# 第七讲 — Krusell-Smith 方法

**异质性主体宏观经济学的计算方法**

孙杰

## 0 · 准备工作

我们导入 `HouseholdStages` 以及常用的数值工具。

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using HouseholdStages
using Printf
using Statistics
using LinearAlgebra: I
using Random: MersenneTwister
using Plots

## 1 · 状态空间爆炸

在 L04 中我们在单一稳态下求解 Aiyagari 模型。在 L05 中我们处理了两个稳态之间的 *确定性* (deterministic) 转移过程。在这两种情形下,一旦我们猜出总量资本 $K$ 的路径,它就是确定无疑的;家庭内层求解从不需要对未来的 $K$ 做积分。

**现在** 加入一个总量生产率冲击 $Z_t$ —— 全要素生产率 (TFP) 服从 $\{Z_{\text{bad}}, Z_{\text{good}}\}$ 上的马尔可夫链。价格 $r_t, w_t$ 依赖于 $(Z_t, K_t)$。关键在于,**家庭无法仅凭 $K_t$ 预测 $K_{t+1}$** —— $K_{t+1}$ 是 *所有* 家庭储蓄决策的积分,而这些决策取决于整个财富分布 $\Lambda_t$。

因此,主体自身的状态空间必须包含 $\Lambda_t$ 才能做出理性决策:
$$V(b_i, z_i; \Lambda, Z) = \max_{c_i, b_i'} u(c_i) + \beta\, \mathbb{E}[V(b_i', z_i'; \Lambda', Z') \mid z_i, Z].$$

但 $\Lambda_t$ 是无穷维的。在其上把 $V$ 列表存储毫无希望。这就是 **Krusell-Smith 问题** (Krusell-Smith problem)。

## 2 · K-S 的应对之道

Krusell 和 Smith (1998) 提出了一个 **有限理性假设** (bounded-rationality assumption):主体不去追踪整个分布 $\Lambda_t$。他们只追踪它的一阶矩 —— 总量资本 $K_t$ —— 并通过一个参数化的 **近似运动定律** (Approximate Law of Motion, ALM) 从 $(K_t, Z_t)$ 预测下一期的 $K$:
$$\log K_{t+1} = a_0(Z_t) + a_1(Z_t) \log K_t.$$

给定 $(a_0, a_1)$,家庭的状态退化为 $(b, y, Z, K)$ —— 有限维的,可以用价值函数迭代 (VFI) 求解。**但** 主体对 $K_{t+1}$ 的预期可能与真实的均衡动态不一致。K-S 迭代 (K-S iteration) 修补了这一点:

1. **选取** 一个初始 ALM $(a_0^{(0)}, a_1^{(0)})$。
2. **求解** 当前 ALM 下的家庭问题:得到 $V^*$ 和政策函数 $b'^*(b, y, Z, K)$。
3. **模拟** (simulate) 一个长经济,使用实现的 $\{Z_t\}$ 和 *真实* 的总量加总 $K_t = \int b\, \mathrm{d}\Lambda_t$。
4. **重新拟合** $(a_0, a_1)$:对 $\log K_{t+1}$ 关于 $\log K_t$ 做 OLS 回归,按 $Z$ 状态分别进行。
5. **迭代** 直至 $(a_0, a_1)$ 不再变动。

## 3 · 参数

经典 K-S 校准:对数效用,$\beta = 0.96$,$\alpha = 0.36$,$\delta = 0.025$。总量 $Z \in \{0.99, 1.01\}$,具有高持续性。个体 $z \in \{0.07, 1.0\}$(失业/就业)。

为教学速度起见,网格刻意取得很小 —— $N_w = 80$,$N_K = 5$。

In [ ]:
@kwdef struct KSParams
    β::Float64 = 0.96
    γ::Float64 = 1.0
    α::Float64 = 0.36
    δ::Float64 = 0.025
    z_grid::Vector{Float64} = [0.07, 1.0]
    P_z::Matrix{Float64}    = [0.6   0.4;
                               0.05  0.95]
    Z_vals::Vector{Float64} = [0.99, 1.01]
    P_Z::Matrix{Float64}    = [0.875 0.125;
                               0.125 0.875]
    N_w::Int       = 80
    w_min::Float64 = 0.0
    w_max::Float64 = 80.0
    N_K::Int       = 5
    K_min::Float64 = 10.5
    K_max::Float64 = 14.5
end
const ks_params = KSParams()
p = ks_params
@printf "β = %.3f, γ = %.2f, α = %.2f, δ = %.3f\n" p.β p.γ p.α p.δ

## 4 · 价格与有效劳动

带 TFP $Z$ 与有效劳动 $L$ 的 Cobb-Douglas 生产函数:
$$r(Z, K) = \alpha\, Z\, (K/L)^{\alpha-1} - \delta, \qquad w(Z, K) = (1-\alpha)\, Z\, (K/L)^\alpha.$$

有效劳动是 $z$ 在平稳分布下的加权平均 —— 在本模型中是一个常数,因为 $P_z$ 与 $Z$ 无关。

In [ ]:
"""CLAUDE
在马尔可夫链 `P_z` 下,`z_grid` 关于平稳分布的加权平均。
"""
function ks_effective_labor(P_z::AbstractMatrix, z_grid::AbstractVector)
    n = size(P_z, 1)
    A = P_z' - I(n)
    A[end, :] .= 1.0
    rhs = zeros(n); rhs[end] = 1.0
    π = A \ rhs
    return sum(z_grid .* π)
end

function ks_prices(K::Real, Z::Real, p::KSParams)
    L = ks_effective_labor(p.P_z, p.z_grid)
    r = p.α * Z * (K/L)^(p.α - 1) - p.δ
    w = (1 - p.α) * Z * (K/L)^p.α
    return (; r, w)
end

L_eff = ks_effective_labor(p.P_z, p.z_grid)
@printf "L_eff = %.4f\n" L_eff

## 5 · $Z = 1$ 时的 Aiyagari 确定性稳态

在加入总量不确定性 (aggregate uncertainty) 之前,先利用 L04 中关于 $K$ 的逐次逼近 (tatonnement) 找到 $Z = 1$ 时的确定性稳态 $\bar K$。它将作为 §6 中 K 网格的自然"中心",同时作为 K-S 模拟的初始 $\Lambda_0$。

In [ ]:
_u_crra(c, ::Union{Val{1}, Val{1.0}}) = log(c)
_u_crra(c, ::Val{σ}) where σ = (c^(1 - σ)) / (1 - σ)
u_crra(c, valσ::Val) = c < 0 ? -Inf : _u_crra(c, valσ)

"""CLAUDE
构造一个 2D (财富, z) 的家庭链,其收入阶段直接从 env 中读取 r 和 w。
用于 Z = 1 时的 Aiyagari 逐次逼近,以及之后模拟过程中在已实现
(Z_t, K_t) 下作为前向迭代器。
"""
function ks_household_2d(p::KSParams)
    layout = StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
        StateAxis(:z, p.z_grid),
    )
    z_shock = MarkovStage(layout; axis=:z, transition=p.P_z)
    income = WealthChangeStage(layout; wealth_post=(cell; env) -> (1 + env.r) * cell.wealth + env.w * cell.z)
    savings = ConsumptionSavingsStage(layout; β=p.β, utility=(cell, c; env) -> u_crra(c, Val(p.γ)), monotone_search=:divide_conquer)
    return define_moments!(z_shock ∘ income ∘ savings; K_supplied=at_end(integrand=:wealth, reduce=sum))
end

"""CLAUDE
在 Z = 1 下使用 `ks_household_2d` 对 K 进行阻尼逐次逼近。返回收敛后的
K 以及 2D 分布 Λ_2d,后者作为 K 网格中心和 K-S 模拟的初始条件。
由于 K_bar 仅作为网格中心,1e-2 的宽松相对容差已足够。
"""
function aiyagari_steady_state_at_Z(p::KSParams; Z::Float64=1.0, verbose::Bool=true)
    hh = ks_household_2d(p)
    K = 12.0
    V, Λ = nothing, nothing
    for iter in 1:200
        env = make_env(hh; ks_prices(K, Z, p)...)
        kw = (; lambda_tol=1e-5, lambda_maxiter=50_000)
        res = isnothing(V) ? solve_steady_state_given_env!(hh, env; kw...) :
              solve_steady_state_given_env!(hh, env; V_init=V, Λ_init=Λ, kw...)
        V, Λ = res.V, res.Λ
        K_sup = res.moments.K_supplied
        K_err = abs(K_sup - K) / K
        verbose && @printf "  iter %d: K = %.3f → K_supplied = %.3f, err = %.5f\n" iter K K_sup K_err
        K_err <= 1e-2 && return (; K, Λ, V, iters=iter)
        K = 0.95 * K + 0.05 * K_sup
    end
    error("aiyagari_steady_state_at_Z: did not converge")
end

println("在 Z = 1.0 处寻找确定性 Aiyagari 稳态...")
@time det_ss = aiyagari_steady_state_at_Z(p; verbose=false)
@printf "K̄ = %.4f,共 %d 次迭代\n" det_ss.K det_ss.iters
K_bar = det_ss.K
Λ_0   = det_ss.Λ

## 6 · 把家庭模块拆成五个阶段

一个时期内的时间顺序:

1. **`z_shock`** —— 个体生产率 $z_i^t \to z_i^{t+1}$(`:z` 上的马尔可夫链)。
2. **`income`** —— 在该单元当前的 $(Z, K)$ 下,以 $r, w$ 更新财富。
3. **`savings`** —— 在财富网格上选择 $b_i^{t+1}$。
4. **`K_evolve`** —— 在主体的 ALM 下确定性地把 $K_t \to K_{t+1}$,在 K 网格上做线性插值。
5. **`Z_shock`** —— $Z_t \to Z_{t+1}$(马尔可夫)。

`K_evolve` 和 `Z_shock` 放在链条 *末端*,这样读取价格的阶段(第 2 步)可以通过单元坐标看到本期开始时的 $(Z, K)$。

In [ ]:
function ks_household(p::KSParams)
    layout = StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
        StateAxis(:z,      p.z_grid),
        StateAxis(:Z_idx,  [1, 2]),
        StateAxis(:K,      continuous_grid(p.K_min, p.K_max; length=p.N_K)),
    )
    z_shock = MarkovStage(layout; axis=:z, transition=p.P_z)
    income = WealthChangeStage(layout; wealth_post=(cell; env) -> begin
        Z = p.Z_vals[cell.Z_idx]
        pr = ks_prices(cell.K, Z, p)
        (1 + pr.r) * cell.wealth + pr.w * cell.z
    end)
    savings = ConsumptionSavingsStage(layout; β=p.β, utility=(cell, c; env) -> u_crra(c, Val(p.γ)), monotone_search=:divide_conquer)
    K_evolve = WealthChangeStage(layout; wealth_axis=:K, wealth_post=(cell; env) -> begin
        a0 = env.a0[cell.Z_idx]
        a1 = env.a1[cell.Z_idx]
        exp(a0 + a1 * log(cell.K))
    end)
    Z_shock = MarkovStage(layout; axis=:Z_idx, transition=p.P_Z)
    chain = z_shock ∘ income ∘ savings ∘ K_evolve ∘ Z_shock
    return define_moments!(chain; K_supplied=at_end(integrand=:wealth, reduce=sum))
end

hh = ks_household(p)
N_w, N_z, N_Z, N_K = layout_size(first(hh.spec.stages).input_layout)
@printf "布局:%d × %d × %d × %d = %d 个单元\n" N_w N_z N_Z N_K (N_w*N_z*N_Z*N_K)

## 7 · 在候选 ALM 下的首次 VFI

初始猜测:常数 ALM —— 主体相信 $K_{t+1}$ 始终等于 $\bar K$,与 $(Z, K)$ 无关。也就是说,对两个 $Z$ 状态都有 $a_0(Z) = \log \bar K$ 与 $a_1(Z) = 0$。在该规则下求解 4D VFI。

In [ ]:
a0_init = [log(K_bar), log(K_bar)]
a1_init = [0.0, 0.0]
env0 = make_env(hh; a0=a0_init, a1=a1_init)
println("在初始 ALM 下求解 4D 家庭 VFI...")
@time ss = solve_steady_state_given_env!(hh, env0; lambda_tol=1e-5, lambda_maxiter=50_000)
@printf "VFI 迭代次数:%d,Λ 迭代次数:%d,ΣΛ = %.10f\n" ss.history.vfi_iters ss.history.lambda_iters sum(ss.Λ)

## 8 · 在已实现 $(Z_t, K_t)$ 下的模拟

我们把 $\Lambda_t$ 维护为 $(b, z)$ 上的 **二维** 分布,并把 $(Z_t, K_t)$ 作为已实现的标量来追踪。每一期:

1. 计算 $K_t = \sum_{b,z} b \cdot \Lambda_t[b, z]$。
2. 把 $\Lambda_t$ 抬升 (lift) 为集中在 $(Z_t, K_t)$ 处的 4D 分布 —— 质量份额按邻近的两个 K 网格点重新分配。
3. **只运行链条的前三个阶段** (`z_shock ∘ income ∘ savings`) 作用于 4D 分布。这绕过了链条中的 `K_evolve` 与 `Z_shock`,否则它们会强加主体对 K/Z 动态的 *信念* 而非 *实现* 的动态。
4. 把 4D 结果对 $(Z, K)$ 做边缘化,得回 $(b, z)$ 上的 $\Lambda_{t+1}$。
5. 从马尔可夫链中抽取 $Z_{t+1} \mid Z_t$。

第 3 步正是链条机制仍在发挥作用之处 —— `income` 阶段中基于份额的财富再分配以及 `savings` 阶段中的政策函数查找,都不是手工易于重写的。

In [ ]:
"""CLAUDE
构造仅包含前三个阶段(z_shock、income、savings)的 4D 链条。用于在
已实现 (Z_t, K_t) 下推进 Λ,绕开 K_evolve/Z_shock 中的主体信念
K/Z 动态。
"""
function ks_household_sim(p::KSParams)
    layout = StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
        StateAxis(:z,      p.z_grid),
        StateAxis(:Z_idx,  [1, 2]),
        StateAxis(:K,      continuous_grid(p.K_min, p.K_max; length=p.N_K)),
    )
    z_shock = MarkovStage(layout; axis=:z, transition=p.P_z)
    income = WealthChangeStage(layout; wealth_post=(cell; env) -> begin
        Z = p.Z_vals[cell.Z_idx]
        pr = ks_prices(cell.K, Z, p)
        (1 + pr.r) * cell.wealth + pr.w * cell.z
    end)
    savings = ConsumptionSavingsStage(layout; β=p.β, utility=(cell, c; env) -> u_crra(c, Val(p.γ)), monotone_search=:divide_conquer)
    return z_shock ∘ income ∘ savings
end

"""CLAUDE
在包夹 K_t 的两个 K 网格点之间的线性插值权重。返回 (i_lo, i_hi, w_lo),
其中 w_lo 的质量份额分到网格点 i_lo,剩余 1-w_lo 分到 i_hi。
"""
function _K_neighbours(K_grid::AbstractVector, K_t::Real)
    K_t_c = clamp(K_t, K_grid[1], K_grid[end])
    i_hi = findfirst(>=(K_t_c), K_grid)::Int
    if i_hi == 1
        return (1, 1, 1.0)
    end
    i_lo = i_hi - 1
    w_lo = (K_grid[i_hi] - K_t_c) / (K_grid[i_hi] - K_grid[i_lo])
    return (i_lo, i_hi, w_lo)
end

"""CLAUDE
使用 4D 模拟链 `hh_sim` 把 Λ_2d 向前推进一期。`hh_sim` 必须已落座好
其核函数(先调用 `backward!` 从收敛的 V* 填充储蓄政策)。返回
Λ_2d_{t+1} 与 K_t。
"""
function ks_simulate_step(hh_sim::ChainStage, p::KSParams, Λ_2d::AbstractMatrix, Z_t_idx::Int, K_t::Real)
    K_grid = axisvalues(first(hh_sim.spec.stages).input_layout.axes[4])
    i_lo, i_hi, w_lo = _K_neighbours(K_grid, K_t)
    N_w, N_z = size(Λ_2d)
    N_Z = length(p.Z_vals)
    N_K = length(K_grid)
    # 抬升到 4D,将 K_t 的质量份额在 i_lo 与 i_hi 之间分配。
    Λ_4d = zeros(N_w, N_z, N_Z, N_K)
    for z in 1:N_z, b in 1:N_w
        m = Λ_2d[b, z]
        iszero(m) && continue
        Λ_4d[b, z, Z_t_idx, i_lo] += m * w_lo
        if i_lo != i_hi
            Λ_4d[b, z, Z_t_idx, i_hi] += m * (1 - w_lo)
        end
    end
    Λ_4d_next = forward!(hh_sim, Λ_4d)
    # 将 (Z, K) 边缘化回 (b, z)。
    Λ_2d_next = dropdims(sum(Λ_4d_next; dims=(3, 4)); dims=(3, 4))
    return Λ_2d_next
end

"""CLAUDE
模拟 T 期。调用方必须事先针对某个 env 调用过 `backward!(hh_sim, V_end,
env)`,以便储蓄政策已落座。返回长度为 T 的 (K_path, Z_idx_path)。
"""
function ks_simulate(hh_sim::ChainStage, p::KSParams, T::Int; Λ_init::AbstractMatrix, Z_init_idx::Int=1)
    rng = MersenneTwister(42)
    wgrid = axisvalues(first(hh_sim.spec.stages).input_layout.axes[1])
    Λ = copy(Λ_init)
    Z_idx = Z_init_idx
    K_path     = zeros(T)
    Z_idx_path = zeros(Int, T)
    for t in 1:T
        K_t = sum(wgrid .* sum(Λ; dims=2))
        K_path[t]     = K_t
        Z_idx_path[t] = Z_idx
        Λ = ks_simulate_step(hh_sim, p, Λ, Z_idx, K_t)
        u = rand(rng)
        Z_idx = u < p.P_Z[Z_idx, 1] ? 1 : 2
    end
    return (; K_path, Z_idx_path)
end

**落座模拟链。** 在 `ks_simulate_step` 能使用储蓄政策之前,我们需要通过 `backward!` 填充链条的储蓄核函数。我们用 §7 中收敛的 V 作为终端价值,在一个有代表性的 env 下对模拟链做后向求解 —— 这会在收敛的家庭感知下把储蓄政策具体化。

In [ ]:
hh_sim = ks_household_sim(p)
# 后向求解会填充储蓄 kernel.policy。除了使用单元坐标 (Z_idx, K) 的闭包之外,
# receipt/savings 阶段不直接读取 env —— 但这些闭包绑定的阶段是从单元的 (Z, K)
# 中取价格,而非从 env 中读取。因此一个空 env 已足够。
backward!(hh_sim, copy(ss.V), (;))
@printf "ks_household_sim 后向求解完成;policy 形状 = %s\n" string(size(hh_sim.buffer.stages[3].kernel.policy))

## 9 · 常数 ALM 下的首次模拟

使用收敛后的 4D 政策与确定性稳态下的 $\Lambda_0$。模拟 $T = 2000$ 期,烧入期 (burn-in) 500 期。

In [ ]:
T_sim  = 2000
T_burn = 500
println("在初始(常数)ALM 下模拟 $T_sim 期...")
@time sim0 = ks_simulate(hh_sim, p, T_sim; Λ_init=Λ_0, Z_init_idx=1)
@printf "K 路径:最小 = %.3f,均值 = %.3f,最大 = %.3f,标准差 = %.3f\n" minimum(sim0.K_path) mean(sim0.K_path) maximum(sim0.K_path) std(sim0.K_path)
@printf "Z = good 的频率:%.3f\n" (sum(sim0.Z_idx_path .== 2) / length(sim0.Z_idx_path))

**直观图示。** $K_t$ 随 $Z_t$ 一同波动。在 *常数* ALM 下,家庭认为 $K_{t+1}$ 永远等于 $\bar K$,但现实中 $K$ 与 $Z$ 同向变动。

In [ ]:
window = (T_burn + 1):(T_burn + 200)
plt_K = plot(window, sim0.K_path[window]; lw=2, label="K_t",
             xlabel="period t", ylabel="K_t",
             title="K-path under initial (constant) ALM",
             size=(700, 320))
hline!(plt_K, [K_bar]; color=:gray, ls=:dash, label="K̄ (initial ALM)")
plt_Z = plot(window, [p.Z_vals[sim0.Z_idx_path[t]] for t in window];
             lw=2, color=:firebrick, label="Z_t",
             xlabel="period t", ylabel="Z_t",
             title="Realised TFP path", size=(700, 160))
plot(plt_K, plt_Z; layout=(2, 1), size=(700, 480))

## 10 · 重新拟合 ALM

**按 $Z_t$ 分组**,分别把 $\log K_{t+1}$ 对 $\log K_t$ 回归。OLS 系数就是更新后的 $(a_0(Z), a_1(Z))$。

In [ ]:
"""CLAUDE
在 Z_t == z_idx 且 t 落在 `window` 中的样本上,把 log K_{t+1} 对
(1, log K_t) 做 OLS 回归。返回 (a0, a1, R², n)。
"""
function regress_ALM(K_path::AbstractVector, Z_idx_path::AbstractVector, z_idx::Int, window::AbstractUnitRange)
    rows = [t for t in window if Z_idx_path[t] == z_idx]
    isempty(rows) && error("regress_ALM: no observations for z_idx = $z_idx")
    x = log.(K_path[rows])
    y = log.(K_path[rows .+ 1])
    x̄, ȳ = mean(x), mean(y)
    Sxx = sum((x .- x̄).^2)
    Sxy = sum((x .- x̄) .* (y .- ȳ))
    β̂  = Sxy / Sxx
    α̂  = ȳ - β̂ * x̄
    ŷ  = α̂ .+ β̂ .* x
    ss_res = sum((y .- ŷ).^2)
    ss_tot = sum((y .- ȳ).^2)
    R²     = 1 - ss_res / ss_tot
    return (; a0=α̂, a1=β̂, R², n=length(rows))
end

window_fit = (T_burn + 1):(T_sim - 1)
fit_bad  = regress_ALM(sim0.K_path, sim0.Z_idx_path, 1, window_fit)
fit_good = regress_ALM(sim0.K_path, sim0.Z_idx_path, 2, window_fit)
@printf "Z = bad  (Z=%.2f): a0 = %+.4f, a1 = %+.4f, R² = %.4f  (n = %d)\n" p.Z_vals[1] fit_bad.a0  fit_bad.a1  fit_bad.R²  fit_bad.n
@printf "Z = good (Z=%.2f): a0 = %+.4f, a1 = %+.4f, R² = %.4f  (n = %d)\n" p.Z_vals[2] fit_good.a0 fit_good.a1 fit_good.R² fit_good.n

## 11 · K-S 外层迭代

把 VFI(§7)、模拟(§8–9)和重新拟合(§10)封装到一个外层循环中。当 ALM 系数不再变动时停止。

In [ ]:
function ks_iterate(p::KSParams, K_bar::Float64, Λ_0::AbstractMatrix; maxiter::Int=15)
    hh     = ks_household(p)
    hh_sim = ks_household_sim(p)
    T_sim  = 2000
    T_burn = 500

    a0 = [log(K_bar), log(K_bar)]
    a1 = [0.0, 0.0]
    R²_hist   = NTuple{2, Float64}[]
    sim_final = nothing

    for it in 1:maxiter
        env = make_env(hh; a0, a1)
        _ss = solve_steady_state_given_env!(hh, env; lambda_tol=1e-5, lambda_maxiter=50_000)
        # 用更新后的 V_ss 重新落座模拟链的政策。
        backward!(hh_sim, copy(_ss.V), (;))

        sim = ks_simulate(hh_sim, p, T_sim; Λ_init=Λ_0, Z_init_idx=1)
        window_fit = (T_burn + 1):(T_sim - 1)
        fit_bad  = regress_ALM(sim.K_path, sim.Z_idx_path, 1, window_fit)
        fit_good = regress_ALM(sim.K_path, sim.Z_idx_path, 2, window_fit)

        a0_new = [fit_bad.a0, fit_good.a0]
        a1_new = [fit_bad.a1, fit_good.a1]
        Δ = maximum(abs.(vcat(a0_new .- a0, a1_new .- a1)))
        @printf "  iter %2d: R² = (%.5f, %.5f), Δ = %.5f, K 均值 = %.3f\n" it fit_bad.R² fit_good.R² Δ mean(sim.K_path[window_fit])
        push!(R²_hist, (fit_bad.R², fit_good.R²))
        sim_final = sim

        a0 = 0.5 .* a0 .+ 0.5 .* a0_new
        a1 = 0.5 .* a1 .+ 0.5 .* a1_new

        Δ < 1e-3 && return (; a0, a1, R²_hist, sim=sim_final, iters=it)
    end
    error("ks_iterate: did not converge")
end

println("运行 K-S 外层迭代...")
@time res_ks = ks_iterate(p, K_bar, Λ_0; maxiter=18)
@printf "✓ 在 %d 次外层迭代中收敛。\n" res_ks.iters

## 12 · 重点结论 —— $R^2 > 0.996$

收敛后的 ALM 回归 $R^2$ 几乎等于 1。收敛政策所蕴含的家庭模块动态,可以被一条对数线性规则 *几乎完美地* 刻画。

这就是 **近似加总** (approximate aggregation):在这一基线校准下,财富分布几乎不影响 $K$ 的动态。仅知道 $K_t$ 与 $Z_t$ 的主体,对 $K_{t+1}$ 的预测能力与知道完整 $\Lambda_t$ 的主体几乎一样好。

为什么?关键驱动因素是 Cobb-Douglas 生产函数,加上相对耐心的主体和 CRRA 效用 —— 家庭的储蓄政策在相关范围内近似为关于财富的线性函数,因此 *积分*(总量 $K$)只由总量 $K$ 本身决定,而不依赖更高阶矩。Krusell & Smith (1998) §3.2 对此有明确讨论。

In [ ]:
@printf "收敛后的 ALM 系数:\n"
@printf "  Z = bad  (Z=%.2f): log K_{t+1} = %+.4f + %+.4f log K_t\n" p.Z_vals[1] res_ks.a0[1] res_ks.a1[1]
@printf "  Z = good (Z=%.2f): log K_{t+1} = %+.4f + %+.4f log K_t\n" p.Z_vals[2] res_ks.a0[2] res_ks.a1[2]
final_R² = res_ks.R²_hist[end]
@printf "最终 R²:(bad: %.5f, good: %.5f)\n" final_R²[1] final_R²[2]

In [ ]:
sim   = res_ks.sim
wfit  = (T_burn + 1):(T_sim - 1)
mask_bad  = sim.Z_idx_path[wfit] .== 1
mask_good = sim.Z_idx_path[wfit] .== 2
x_bad  = log.(sim.K_path[wfit][mask_bad])
y_bad  = log.(sim.K_path[wfit .+ 1][mask_bad])
x_good = log.(sim.K_path[wfit][mask_good])
y_good = log.(sim.K_path[wfit .+ 1][mask_good])

plt_scatter = plot(title="ALM fit at K-S convergence",
                   xlabel="log K_t", ylabel="log K_{t+1}",
                   size=(640, 420), legend=:topleft)
scatter!(plt_scatter, x_bad,  y_bad;  ms=2, alpha=0.3, label="Z = bad")
scatter!(plt_scatter, x_good, y_good; ms=2, alpha=0.3, label="Z = good", color=:firebrick)
x_grid = range(min(minimum(x_bad), minimum(x_good)),
               max(maximum(x_bad), maximum(x_good)); length=50)
plot!(plt_scatter, x_grid, res_ks.a0[1] .+ res_ks.a1[1] .* x_grid;
       lw=2, color=:steelblue, label="ALM (Z=bad)")
plot!(plt_scatter, x_grid, res_ks.a0[2] .+ res_ks.a1[2] .* x_grid;
       lw=2, color=:firebrick, label="ALM (Z=good)")

## 13 · Den Haan 批评

$R^2$ 是在许多期上平均的,可能掩盖系统性偏误。Den Haan (2010) 提出了一种更敏锐的诊断:**仅用** ALM 把 $K$ 向前模拟许多期(不做主体的重新加总),再与真实路径作比较。

如果 ALM 是对动态的忠实概括,那么这两条路径应当大致重合。在我们的基线校准下,它们的确如此 —— 在数百期内仅差几个百分点。

In [ ]:
function ks_alm_forward(K_0::Real, Z_idx_path::AbstractVector, a0, a1)
    T = length(Z_idx_path)
    K_alm = zeros(T)
    K_alm[1] = K_0
    for t in 1:(T - 1)
        z = Z_idx_path[t]
        K_alm[t+1] = exp(a0[z] + a1[z] * log(K_alm[t]))
    end
    return K_alm
end

K_alm_path = ks_alm_forward(sim.K_path[T_burn + 1], sim.Z_idx_path[T_burn + 1 : end],
                            res_ks.a0, res_ks.a1)
window_dh = 1:min(800, length(K_alm_path))
true_path = sim.K_path[T_burn + 1 : T_burn + length(K_alm_path)]
max_err_pct = 100 * maximum(abs.(K_alm_path[window_dh] .- true_path[window_dh]) ./ true_path[window_dh])
@printf "Den Haan 诊断:在 %d 期内,最大 |%%误差| = %.2f%%\n" length(window_dh) max_err_pct

plot(window_dh, true_path[window_dh]; lw=2, label="True K_t (full aggregation)",
     xlabel="period (post-burn-in)", ylabel="K",
     title="Den Haan diagnostic", size=(720, 380))
plot!(window_dh, K_alm_path[window_dh]; lw=2, ls=:dash,
      label="ALM-only K_t (no aggregation)", color=:firebrick)

**对图的解读。** 虚线(仅 ALM)路径与实线(真实)路径之间的差距维持在几个百分点以内。在这一校准下,K-S 的有限理性假设几乎是精确的 —— ALM 不仅在描述意义上准确,在 *规范意义* 上也足够好:使用 ALM 的主体所做的决策,与使用完整分布的主体所做的决策几乎无法区分。

这正是让 K-S 一举成名的标志性结果。**但是**,近似加总是 *这一* 基线校准的性质,而非异质性主体 (HA) 模型的一般特征。修改校准 —— 非 CRRA 偏好、职业选择、厚尾收入冲击、个体边际消费倾向 (MPC) 的巨大差异 —— 线性 ALM 就可能失效。这正是参数化 ALM 失败、**神经网络参数化** (neural-network parameterisations)(下一讲)登场之处。

## 14 · 预告 L08

L07 给出了一个 K-S 基线方法,它之所以可行,是因为近似加总。L08 把它一般化:

- **用** $V_\theta(b_i, z_i, Z, K)$ 这一神经网络 **取代** 参数化 ALM。不再依赖对数线性假设。
- **训练** $V_\theta$ 以最小化 Bellman 残差,采样覆盖个体与总量状态(即"汇集 Bellman" (Pooled Bellman) 残差)。
- **抛弃** ALM 与那一步回归。主体的预期都内化在 $V_\theta$ 之中。
- **要点:**"Continuation Value is All You Need" —— 同样的想法,以非参数化方式参数化。

CVIAYN 继承了 K-S 基于阶段的结构(同样是 `z_shock ∘ income ∘ savings ∘ K_evolve ∘ Z_shock` 链条),但训练的是 $V_\theta$,而不是迭代"ALM 加 VFI"的循环。